In [2]:

from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim 
import torch.nn.functional as F
import os
from torch.utils.data import DataLoader 
from torch.utils.data import Dataset
import torchvision.transforms as transforms



In [4]:

class SignLanguageDataSet(Dataset):
    def __init__(self, root_directory, shape, train=True, transform=None):
        self.root_dir = root_directory
        self.shape = shape
        self.transform = transform if transform else self.get_default_transform()
        self.image_paths = []
        self.labels = []

        # Load image paths and labels using helper function
        self.load_dataset(train)

    def get_default_transform(self):
        return transforms.Compose([
            transforms.Resize(self.shape),
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) #channel wise normalization
        ])

    def load_dataset(self, train=True):
        for label in range(10): #as its from 0 to 9
            digits_folder = os.path.join(self.root_dir, str(label))
            image_files = [img for img in os.listdir(digits_folder) if img.endswith('.JPG')]
            split_index = int(0.7 * len(image_files))

            if train:
                selected_images = image_files[:split_index]  #If we are preparing the train dataset, we take the first 70%.
            else:
                selected_images = image_files[split_index:] #If we are preparing the test dataset, we take the remaining 30%.

            for img_name in selected_images:
                full_path = os.path.join(digits_folder, img_name)
                self.image_paths.append(full_path)
                self.labels.append(label)          #saving the complete paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img_path = self.image_paths[index]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[index]

        if self.transform:
            image = self.transform(image)

        return image, label


In [12]:
batch_size=64

train_dataset = SignLanguageDataSet(

    root_directory='Sign-Language-Digits-Dataset/Dataset',

    shape=(64,64),
    
    train=True
)
test_dataset = SignLanguageDataSet(

    root_directory='Sign-Language-Digits-Dataset/Dataset',

    shape=(64,64),
    
    train=False
)
train_loader = DataLoader( # 1)extract the dataset from source and 3) load it in form of batches.

    dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(

    dataset=test_dataset, batch_size=batch_size, shuffle=False
)

print(train_dataset.shape)
print(test_dataset.shape)



(64, 64)
(64, 64)


In [ ]:
class NN(nn.Module):  #1. nn.Module is the base class of all neural network models,

                      # we need to extend nn.Module and defined our subclass like NN

                      # our model should be the subclass of nn.Module

                      #2. define layers of subclass

                      #3. implement forward()

 

    def __init__(self, input_size, num_classes): # constructor of NN with its attributes

        super(NN, self).__init__() # calling constructor of base class

                                    # create two layer NN, first layer with 50 neural and second/output layers with 10 neuraons   

         #self.fc1.weight.shape =  50,input_size
        self.fc1 = nn.Linear(input_size, 256) #input size is 64*64*3
        self.fc2 = nn.Linear(256, 64)
        self.fc3= nn.Linear(64,num_classes) #num of classes are 10 (0-9)
        # callable objects

    def forward(self, x):  # we must provid imp of forward () of nn.Module in our subclass
        
        #experimented with softmax instead accuracy decreased significantly
       # x = F.softmax(self.fc1(x))
        #x= F.softmax(self.fc2(x))
        #with sigmoid accuracy remained nearly same
        #x=F.sigmoid(self.fc1(x))
        #x=F.sigmoid(self.fc2(x))

        x=F.relu(self.fc1(x))
        x=F.relu(self.fc2(x))

        x=self.fc3(x)  #no activation function just logits , beacuse in cross entropy softmax will automatically applied
         # //can do F.softmax(self.fc1(x))

        #x = self.fc2(x)  #         x = F.softmax(self.fc2(x), dim=1)

        #x = F.softmax(self.fc2(x))#, dim=1)

        return x

 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

 

input_size =12288       #64*64*3  = 12288 size of RGB image (3 channels)

num_classes = 10

learning_rate = 0.001

num_epochs = 6

 

# create NN object and move it to device

 

'''

When we initialize the model the weights and biases

of the model will be initialized under the hood of PyTorch

and if you want a customized weight initialization it can be added in the NN class.

'''

 

model = NN(input_size=input_size, num_classes=num_classes).to(device)

 

'''

The standard loss function for classifications tasks in PyTorch is the CrossEntropyLoss()

which applies the softmax functionand negative log likelihood given the predictions

of the model and data labels.

'''

criterion = nn.CrossEntropyLoss()

 

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

 

 

for epoch in range(num_epochs):

    print(f"Epoch: {epoch}")

    for batch_idx, (data, targets) in enumerate(train_loader):

        data = data.to(device=device)

        targets = targets.to(device=device)

        data = data.reshape(data.shape[0], -1) #[64,64*64*3]=[64, 12288]

        #print (data.shape) #[64,784]

 

        # forward propagation

        scores = model(data) #automatically call the forward method,

                                #as model is a callable object

        loss = criterion(scores, targets) # compute cost/loss on 64 example

 

        # zero previous gradients

        optimizer.zero_grad()

       

        # back-propagation

        loss.backward()

 

        # gradient descent or adam step

        optimizer.step()

 

       


Epoch: 0
Epoch: 1
Epoch: 2
Epoch: 3
Epoch: 4
Epoch: 5


In [27]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0
    model.eval() # 1. our model deactivates all the layers (eg.batch normalization/dropout)
    with torch.no_grad(): #2.  not make computational graph
        for x, y in loader:
            #print (x.shape)
            x = x.to(device=device)
            y = y.to(device=device)
           
            x = x.reshape(x.shape[0], -1)
            print(x.shape)
            #print (y.shape)
            
            scores = model(x)
            print(scores.shape)
                      
            _, predictions = scores.max(1) #. it return max value and its index, 1 mean see column-wise 
            
            num_correct += (predictions == y).sum() # compare prediction with y, if equal sum them to count the number of same values
            num_samples += predictions.size(0)  #64, get no of samples
            break  # just to see the results for a single patch
        print(
            f"Got {num_correct} / {num_samples} with accuracy"
            f" {float(num_correct) / float(num_samples) * 100:.2f}"
        )
print ("Test accuracy: ")
check_accuracy(test_loader, model)


print ("Train accuracy: ")
check_accuracy(train_loader, model)



Test accuracy: 
torch.Size([64, 12288])
torch.Size([64, 10])
Got 53 / 64 with accuracy 82.81
Train accuracy: 
torch.Size([64, 12288])
torch.Size([64, 10])
Got 50 / 64 with accuracy 78.12
